# Recherche d'information dans la presse française de 1914

## Récupération des données


In [ ]:
!git clone https://github.com/nzmonzmp/dataset-article-fr-light.git

In [ ]:
!ls dataset-article-fr-light/le_figaro/1914/

## Corpus

Ce corpus est composé d'articles de la presse française de l'année 1914.

Un dossier par journal, un dossier contenant un fichier JSON par date de parution.

Les fichiers JSON contiennent plusieurs champs, dont trois qui nous intéresseront pour la suite de ce travail :

- `contentAsText`, une liste de chaînes de caractères (les paragraphes sont délimités dans ces chaînes par des sauts de ligne — `\n`)
- `date`, la date de parution du journal, représentée par une liste contenant une chaîne de caractères (si la variable `content` contient le fichier JSON chargé, on y accède par `content["date"][0]`)
- `title`, le nom du journal, représenté par une liste contenant une chaîne de caractères (si la variable `content` contient le fichier JSON chargé, on y accède par `content["title"][0]`)

Implémentez une fonction qui prend en entrée le chemin du corpus et qui retourne un tableau `numpy.ndarray` à 3 colonnes :

- la première colonne contient les paragraphes des éditions du corpus (on a donc une ligne dans le tableau par paragraphe du corpus), avec respect des conditions suivantes :
  - seulement les paragraphes de plus de 10 mots
  - casse minuscule
  - sans signes de ponctuation
- la deuxième colonne contient le journal dont le paragraphe est issu
- la troisième colonne contient la date de l'édition dont le paragraphe est issu

In [ ]:
# Votre code ici

### Solution

In [ ]:
import datetime
import json
import pathlib
import re

import numpy
import tqdm.notebook


def preprocess_text(text: str) -> str:
  out = re.sub(r"\W+", " ", text)
  return out.lower()


def get_data(directory: pathlib.Path) -> numpy.ndarray:
  data = []
  for edition in tqdm.notebook.tqdm(list(directory.rglob("*.json"))):
    with open(edition, encoding="utf8") as fh:
      content = json.load(fh)
    newspaper = content["title"][0]
    date = datetime.date.fromisoformat(content["date"][0]).strftime("%d/%m/%Y")
    for item in content["contentAsText"]:
      for paragraph in item.split("\n"):
        processed_paragraph = preprocess_text(paragraph)
        if len(processed_paragraph.split()) > 10:
          data.append((processed_paragraph, newspaper, date))
  return numpy.array(data)


data = get_data(pathlib.Path("dataset-article-fr-light"))

## Représentation TF-IDF

Utilisez un [`sklearn.feature_extraction.text.TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) afin de créer la matrice `X` des paragraphes du corpus transformés en vecteurs de poids TF-IDF.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import sklearn.feature_extraction.text

vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
X = vectorizer.fit_transform(data[:, 0])

## Similarité

Implémentez une fonction `best_score(query: str) -> None` qui :

- prend en entrée une requête sous forme de chaîne de caractères
- affiche les journaux et dates des 5 meilleurs résultats au sens de la similarité cosinus

In [ ]:
# Votre code ici

### Solution

In [ ]:
import sklearn.metrics.pairwise


def best_score(query: str) -> None:
  query_vector = vectorizer.transform([query.lower()])
  cosine = sklearn.metrics.pairwise.cosine_similarity(X, query_vector)[:, 0]
  # On utilise l'opposé du tableau cosine dans l'appel à argsort afin d'obtenir
  # les indices du tableau triés par valeurs décroissantes avant d'en récupérer
  # les 5 premiers
  top5 = (-cosine).argsort()[:5]
  for index, (text, newspaper, date), similarity in zip(top5,
                                                        data[top5],
                                                        cosine[top5]):
    print(f"{newspaper:10} du {date} ; "
          f"similarité {similarity:5.2f} ; "
          f"index {index:6} ; "
          f"{text}")

## Recherche

Cherchez les passages les plus pertinent qui parlent de :

- l'attentat contre François-Ferdinand
- guerre contre les allemands

Vérifiez vos résultats en allant regarder le texte des journaux dans `X`

In [ ]:
# Votre code ici

### Solution

In [ ]:
best_score("attentat François-Ferdinand")

In [ ]:
best_score("Guerre allemands")